# DP9 の下準備：7B（Qwen2.5-7B-Instruct）が L4 でどれくらいの速さで動くかを測る

> ## 🔑 先に許可しておくシークレット
> - **`GITHUB_TOKEN`（必須）**：リポジトリを決まったコミットで取ってきます。
> - `HF_TOKEN`（任意）：Qwen は利用許可のいらないモデルなので、なくても動きます。あればダウンロードが安定します。
>
> 左の鍵のマークで、上の2つの「ノートブックからのアクセス」をオンにしてから「すべてのセルを実行」してね。

**これは何？**：本番（DP9）の設計を決める前の **時間計測だけ** です。**ベンチマークの問題は使わず、採点もしません。** 使うのは、これまでの確認と同じ C5-A の2つの文と、この計測のために書いた短い算数の問題4つ（答えの長さを数えるだけ）です。

**確かめること**
- 今の仕組み（土台の道・引き直し・迷いのサイン・自然に引いた別の語・確率 10⁻⁶ に近い語の選び方）が、Qwen の語彙（約15万語）でも Python の実装と同じ答えを出すか
- 7B が L4（24GB）に収まるか、1つの位置の計算にどれくらいかかるか、まとめて何本まで引けるか
- 「種類ごとに別々に引く」（DP8 まで）と「全部の種類を1つのまとまりで引く」（DP9 の候補）の両方で、本番が何時間・何 CU になるか

**使い方**
1. ランタイム → ランタイムのタイプを変更 → **L4**（ハイメモリは不要）。
2. **「すべてのセルを実行」**。モデルのダウンロード（約15GB）を入れて **20〜30分くらい（約0.5〜0.9 CU）** です。
3. 結果は Drive の `DECISION_POINTS/dp9/SMOKE_<時刻>/SMOKE.json`（数字だけ）。終わったら Claude に知らせてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。
- **エラーのとき**：通知の本文にエラーの文面（トレースバック）が入ります。全文は Drive の `DECISION_POINTS/_errors/` にも保存されるので、Claude が直接読めます。

- 通知の頭には 🚀（始まり）／⚠（エラー）／🎉（終わり）が付きます。

**止まる場所**：セル1（シークレット・取得・インストール）／セル2（サインやサンプリングが Python の実装と不一致）／セル4（E2・G1・B1。E2 は 2026-10-17 に「1歩でもずれたら止める」から「32歩中29歩以上で1位が合い、ずれが 1.0 以下」に直しました）。M1（まとめて引くやり方の確認）は止めずに記録だけします。メモリが足りない大きさは、止めずに「入らない」と記録します。

In [ ]:
# Cell 1 — Drive, GPU check, the exact head, install, pin the model revision (Hub API) and download it to local disk.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。エラーのときはエラーの文面も送ります（鍵・トークンらしい文字は伏せ字。全文は Drive の DECISION_POINTS/_errors）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'DP9 時間計測（Qwen2.5-7B・L4・採点なし）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'GodPork - Pork DP9 smoke', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
import os, re as _re, traceback as _tb, datetime as _edt
ERR_SLUG = 'DP9_SMOKE'   # Drive のエラーファイルの名前の頭
_SECRET_RE = _re.compile(r"sk-ant-[A-Za-z0-9_\-]+|github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9]{8,}|hf_[A-Za-z0-9]{8,}|x-access-token:[^@\s]+@")
def _scrub(s):
    s = _SECRET_RE.sub('[伏せ字]', s)
    return '\n'.join(l if len(l) <= 300 else l[:300] + ' …（長いので省略）' for l in s.splitlines())
def _error_text(result):
    e = result.error_in_exec if result.error_in_exec is not None else result.error_before_exec
    tb = ''.join(_tb.format_exception(type(e), e, e.__traceback__))
    first = (getattr(getattr(result, 'info', None), 'raw_cell', '') or '').strip().splitlines()[:1]
    return _scrub('セル: ' + (first[0][:100] if first else '?') + '\n' + tb)
def _save_error(text):
    base = '/content/drive/MyDrive'
    if not os.path.isdir(base):
        return None
    d = os.path.join(base, 'DECISION_POINTS', '_errors')
    os.makedirs(d, exist_ok=True)
    stem = ERR_SLUG + '_' + _edt.datetime.now(_edt.timezone(_edt.timedelta(hours=9))).strftime('%Y%m%d_%H%M%S')
    name, n = stem + '.txt', 1
    while os.path.exists(os.path.join(d, name)):
        n += 1
        name = stem + '_' + str(n) + '.txt'
    with open(os.path.join(d, name), 'w', encoding='utf-8') as fh:
        fh.write('[' + NB_LABEL + ']\n' + text)
    return 'DECISION_POINTS/_errors/' + name
def _short(text, limit=3300):
    b = text.encode('utf-8')
    if len(b) <= limit:
        return text
    return '…（前のほうは省略。全文は Drive）\n' + b[-limit:].decode('utf-8', 'ignore')
def _on_error(result):
    if result.error_in_exec is None and result.error_before_exec is None:
        return
    try:
        text = _error_text(result)
    except Exception as e2:
        text = 'エラーの文面を作れませんでした：' + type(e2).__name__
    try:
        where = _save_error(text)
    except Exception as e3:
        where = None
        text += '\n（Drive に保存できませんでした：' + type(e3).__name__ + '）'
    print('エラーの全文を通知に送ります' + ('（Drive：' + where + '）' if where else ''))
    notify('⚠ 止まりました：エラーが出ています。' + ('全文は Drive の ' + where if where else 'Drive には保存できませんでした')
           + '\n\n' + _short(text), tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

import tempfile
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # allocation only (as DP4-DP8)
drive.mount("/content/drive")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
assert gpu.returncode == 0 and gpu.stdout.strip(), "GPU のランタイムにしてね（L4）"
GPU_NAME = gpu.stdout.strip().split(",")[0]
print("GPU:", gpu.stdout.strip())
if "L4" not in GPU_NAME:
    print("⚠️ このランタイムは L4 ではありません（" + GPU_NAME + "）。動きますが、測った時間は L4 の値になりません。")

EXPECTED_HEAD = "1172e5b6788b6496076fe06169094ffd4e900a6e"
BRANCH = "decision-points"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")
THEME = Path("/content/drive/MyDrive/DECISION_POINTS")
STAMP = dt.datetime.now(dt.timezone(dt.timedelta(hours=9))).strftime("%Y%m%d_%H%M%S")
OUT = THEME / "dp9" / ("SMOKE_" + STAMP)
OUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR = Path("/content/qwen2.5-7b-instruct")

def get_secret(name):
    try:
        v = userdata.get(name)
    except Exception:
        return None
    return str(v).strip() if v and str(v).strip() else None

token = get_secret("GITHUB_TOKEN") or get_secret("GH_TOKEN")
assert token, "Colab のシークレットに GITHUB_TOKEN を登録して、ノートブックからのアクセスをオンにしてね"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
os.chdir('/content')   # セル1 をやり直しても、消すフォルダの中にいないように（2026-10-14）
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
token = None
env.pop("GITHUB_TOKEN", None)
print("exact head:", head)

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "transformers==4.57.6",
                    "huggingface_hub", "pyarrow", "scikit-learn", "scipy"], capture_output=True, text=True)
(OUT / "install_log.txt").write_text(r.stdout + "\n--- STDERR ---\n" + r.stderr)
assert r.returncode == 0, r.stderr[-4000:]
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import transformers, torch
assert transformers.__version__ == "4.57.6", transformers.__version__
from pork import dp9_smoke as S

def atomic_json(p, value):
    p.parent.mkdir(parents=True, exist_ok=True)
    data = json.dumps(value, ensure_ascii=False, indent=1, sort_keys=True) + "\n"
    fd, tmp = tempfile.mkstemp(prefix=p.name + ".", suffix=".tmp", dir=p.parent)
    with os.fdopen(fd, "w", encoding="utf-8") as f:
        f.write(data); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, p)

SMOKE = {"schema": S.SMOKE_SCHEMA, "code_commit": EXPECTED_HEAD, "gpu_name": GPU_NAME, "gpu": gpu.stdout.strip(),
         "model_id": S.MODEL_ID, "dtype": S.DTYPE, "started_jst": STAMP, "python": sys.version.split()[0],
         "torch": torch.__version__, "transformers": transformers.__version__}

# The model revision: the Hub's current commit of the main branch, recorded here; the DP9 gate will pin it.
from huggingface_hub import HfApi, snapshot_download
HF = get_secret("HF_TOKEN")
SMOKE["hf_token_used"] = bool(HF)
info = HfApi().model_info(S.MODEL_ID, token=HF, files_metadata=True)
REVISION = info.sha
def _lfs_sha(sib):
    lfs = getattr(sib, "lfs", None)
    if lfs is None:
        return None
    return lfs.get("sha256") if isinstance(lfs, dict) else getattr(lfs, "sha256", None)
FILES = {s.rfilename: {"size": s.size, "lfs_sha256": _lfs_sha(s)} for s in info.siblings}
t0 = time.time()
snapshot_download(S.MODEL_ID, revision=REVISION, local_dir=str(MODEL_DIR), token=HF,
                  allow_patterns=["*.json", "*.safetensors", "merges.txt", "vocab.json", "tokenizer*"])
SMOKE["download_seconds"] = time.time() - t0
t0 = time.time()
checked = {}
for name, meta in sorted(FILES.items()):
    p = MODEL_DIR / name
    if not p.exists():
        continue
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 24), b""):
            h.update(chunk)
    checked[name] = {"size": p.stat().st_size, "sha256": h.hexdigest(),
                     "matches_hub_lfs": None if meta["lfs_sha256"] is None else h.hexdigest() == meta["lfs_sha256"]}
assert all(v["matches_hub_lfs"] is not False for v in checked.values()), "ダウンロードしたファイルが Hub の記録と違います"
SMOKE["model_revision"] = REVISION
SMOKE["model_files"] = checked
SMOKE["hash_seconds"] = time.time() - t0
CONFIG = json.loads((MODEL_DIR / "config.json").read_text())
SMOKE["config"] = {k: CONFIG.get(k) for k in ("architectures", "vocab_size", "hidden_size", "num_hidden_layers",
                                               "num_attention_heads", "num_key_value_heads", "max_position_embeddings",
                                               "tie_word_embeddings", "use_sliding_window", "torch_dtype")}
atomic_json(OUT / "SMOKE.json", SMOKE)
print("model revision:", REVISION, "| files:", len(checked), "| download", round(SMOKE["download_seconds"]), "s")

In [ ]:
# Cell 2 — GPU vs plain-Python implementations on synthetic logits with ties (vocab 2,000 and Qwen's output size): the
# base path's per-step signals (DP1's step_signals), pure sampling at T 1.0, the alt first-token draw (base token
# removed) and the w_tau choice (closest to 1e-6; top-1 and excluded ids never chosen).  No model.
import math, random, struct
import torch as torch_mod
from pork import pork_t_v1 as pt
from pork import q4b_samplers as qb
from pork import dp1_forks as dp
from pork import dp8_bridge as d8
Bg = pt.backend("torch", "cuda")
V_OUT = int(CONFIG["vocab_size"])

def f32(x):
    return struct.unpack("f", struct.pack("f", x))[0]

def synth(rng, vocab):
    xs = [f32(rng.gauss(0.0, rng.choice((1.0, 2.5, 4.0)))) for _ in range(vocab)]
    for i in range(0, vocab, 97):
        xs[i] = xs[0]
    if rng.random() < 0.3:
        j = rng.randrange(vocab); xs[j] = f32(xs[j] + 12.0)
    if rng.random() < 0.2:
        xs[1] = xs[0] = max(xs)
    if rng.random() < 0.4:
        m = max(xs)
        v = f32(m + math.log(d8.TAU) + rng.choice((0.0, 0.3, -0.3)))
        for j in rng.sample(range(2, vocab), 3):
            xs[j] = v
    return xs

rng = random.Random(S.PARITY_SEED)
names = ("SIGNALS", "PURE", "ALT", "TAU")
total = {k: 0 for k in names}; mism = {k: 0 for k in names}
t0 = time.time()
for vocab, steps in ((2000, 150), (V_OUT, 8)):
    mg = mr = None
    for step in range(steps):
        xs = synth(rng, vocab)
        g = torch_mod.tensor(xs, dtype=torch_mod.float32, device="cuda")
        a, b = dp.step_signals(Bg, g, mg), dp.ref_step_signals(xs, mr)
        total["SIGNALS"] += 1
        if ((a["top1"], a["top2"]) != (b["top1"], b["top2"]) or abs(a["p1"] - b["p1"]) > 1e-9
                or abs(a["p2"] - b["p2"]) > 1e-9 or abs(a["entropy"] - b["entropy"]) > 1e-9):
            mism["SIGNALS"] += 1
        mg, mr = a["mu_after"], b["mu_after"]
        u = rng.random()
        total["PURE"] += 1
        if tuple(int(v) for v in qb.pure_sample(Bg, g, u, 1.0)) != tuple(int(v) for v in qb.ref_pure(xs, u, 1.0)):
            mism["PURE"] += 1
        exa = b["top1"] if step % 2 == 0 else rng.randrange(vocab)
        u2 = rng.random() if step % 7 else 0.9999999999
        aa, ab = d8.alt_sample(Bg, g, u2, 1.0, exa), d8.ref_alt(xs, u2, 1.0, exa)
        total["ALT"] += 1
        if aa != ab or aa == exa:
            mism["ALT"] += 1
        ex = rng.sample(range(vocab), 4)
        ta, tb = d8.tau_choice(Bg, g, b["top1"], ex), d8.ref_tau_choice(xs, b["top1"], ex)
        total["TAU"] += 1
        if (ta["token"], ta["matched"]) != (tb["token"], tb["matched"]) or abs(ta["p"] - tb["p"]) > 1e-9 * tb["p"]:
            mism["TAU"] += 1
SMOKE["parity"] = {"total": total, "mismatches": mism, "vocab_sizes": [2000, V_OUT], "seconds": time.time() - t0,
                   "pass": all(v == 0 for v in mism.values())}
atomic_json(OUT / "SMOKE.json", SMOKE)
print(json.dumps(SMOKE["parity"], indent=1))
assert SMOKE["parity"]["pass"], "サイン・サンプリングの照合が合いません — Claude に見せてね"
print("PARITY PASS")

In [ ]:
# Cell 3 — load the model (bfloat16; the library's default attention for Qwen2 is recorded), and check the tokenizer:
# EOS ids, the excluded ids for w_tau, the chat template (counts only), digit tokens.
from pork import kv_cached_backend as kv
from pork import dp2_real_forks as d2
from pork import dp8_runner as rr
from pork.ct1_runner import torch_fns
from pork.ultimapork_completion_c5a import PROMPTS as C5A_PROMPTS
t0 = time.time()
tokenizer = transformers.AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = transformers.AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), torch_dtype=torch_mod.bfloat16).to("cuda")
model.eval()
SMOKE["load_seconds"] = time.time() - t0
ATTN = model.config._attn_implementation
def set_attn(name):
    try:
        if hasattr(model, "set_attn_implementation"):
            model.set_attn_implementation(name)
        else:
            model.config._attn_implementation = name
    except Exception:
        model.config._attn_implementation = name
    return model.config._attn_implementation == name
EOS = kv.eos_ids(tokenizer, model)
TAU_EXCLUDED = rr.excluded_ids(tokenizer, EOS)
VOCAB = int(model.get_output_embeddings().weight.shape[0])
DEVICE = "cuda"
_single, _batch = torch_fns(model, torch_mod, DEVICE)
Bt = pt.backend("torch", DEVICE)
decode = lambda ids: tokenizer.decode(list(ids), skip_special_tokens=True)
im_start = tokenizer.convert_tokens_to_ids("<|im_start|>")
c5a_ids = [kv.chat_prompt_ids(tokenizer, t) for _, t in C5A_PROMPTS]
own_ids = [kv.chat_prompt_ids(tokenizer, d2.user_message(q)) for q, _ in S.OWN_PROBLEMS]
digits_single = all(len(tokenizer.encode(c, add_special_tokens=False)) == 1 for c in "0123456789")
n12345 = len(tokenizer.encode("12345", add_special_tokens=False))
digit_tokens = sum(d2.has_digit(tokenizer.decode([i])) for i in range(len(tokenizer)))
SMOKE["model"] = {
    "attn_implementation": ATTN, "dtype": str(model.dtype), "vocab_out": VOCAB, "tokenizer_len": len(tokenizer),
    "eos_ids": list(EOS), "tau_excluded_count": len(TAU_EXCLUDED), "special_ids_count": len(tokenizer.all_special_ids),
    "chat_turn_starts_c5a": [sum(int(x == im_start) for x in ids) for ids in c5a_ids],
    "default_system_turn": all(sum(int(x == im_start) for x in ids) == 3 for ids in c5a_ids),
    "prompt_tokens_c5a": [len(x) for x in c5a_ids], "prompt_tokens_own": [len(x) for x in own_ids],
    "digits_are_single_tokens": digits_single, "tokens_in_12345": n12345, "vocab_tokens_with_digit": digit_tokens,
    "gpu_gb_total": torch_mod.cuda.get_device_properties(0).total_memory / 1e9,
    "gpu_gb_after_load": torch_mod.cuda.memory_allocated() / 1e9,
}
assert str(model.dtype) == "torch.bfloat16" and VOCAB == V_OUT
assert all(0 <= x < VOCAB for x in TAU_EXCLUDED)
SMOKE["workers_possible"] = max(1, int((SMOKE["model"]["gpu_gb_total"] - 2.0) // (SMOKE["model"]["gpu_gb_after_load"] + 4.0)))
atomic_json(OUT / "SMOKE.json", SMOKE)
print(json.dumps(SMOKE["model"], indent=1), "\nworkers possible on this GPU:", SMOKE["workers_possible"])

In [ ]:
# Cell 4 — infrastructure checks.  Revised 2026-10-17 12:15 JST after the first run stopped at E2 (Qwen 7B, bfloat16,
# sdpa: the cached step and the full replay differed by up to 0.44 in a logit and the argmax differed at some step of
# both prompts; the old rule asked for agreement at every step, which held for Gemma-2-2b with eager attention).
#   E2: cached step vs full replay without cache, teacher forced (both fed the replay's token), 32 steps per consumed
#       C5-A prompt.  Recorded per prompt: agreeing steps, the largest and the median per-step max |logit diff|, and at
#       each disagreeing step the replay's top-1 minus top-2 logit gap.  Stop only if a prompt agrees at < 29 of 32
#       steps (B1's rule) or its largest difference exceeds 1.0 (a cache error would show as large differences at many
#       steps; bfloat16 rounding near logit 30 is 0.125-0.25 per step).  Also run with eager attention (recorded only).
#   G1: DP8's base-path generator (step_signals' top-1 on the GPU, w_tau at every step) vs plain greedy on the SAME
#       cached logits (must be identical), and the generator run twice (must be identical: determinism), 64 tokens.
#   B1: 32 identical rows batched vs one row, >= 29 of 32 steps (as CT-1 .. DP8).
#   M1 (recorded, not a stop): the merged way vs each prefix alone (>= 29 of 32 per group), and the 16-token chunk's
#       last logits vs the token-by-token path's (largest |diff|, argmax, the top-1 minus top-2 gap).
from pork import quality_pork_q8t as q8

def e2_run(ids, steps=32):
    step = _single()
    seq, past, new = list(ids), None, tuple(ids)
    agree, diffs, gaps = 0, [], []
    for _ in range(steps):
        a, past = step(new, past)
        with torch_mod.no_grad():
            bfull = model(input_ids=torch_mod.tensor([seq], device=DEVICE), use_cache=False).logits[0, -1].float()
        diffs.append(float((a - bfull).abs().max()))
        ta, tb = int(a.argmax()), int(bfull.argmax())
        if ta == tb:
            agree += 1
        else:
            top2 = torch_mod.topk(bfull, 2).values
            gaps.append(float(top2[0] - top2[1]))
        seq.append(tb); new = (tb,)
    diffs_sorted = sorted(diffs)
    return {"steps": steps, "agree_steps": agree, "max_abs_logit_diff": max(diffs),
            "median_step_max_abs_logit_diff": diffs_sorted[len(diffs) // 2], "gap_at_disagreements": gaps}

E2 = []
for (name, _), ids in zip(C5A_PROMPTS, c5a_ids):
    r_ = e2_run(ids)
    r_.update({"prompt": name, "attention": ATTN, "pass": r_["agree_steps"] >= 29 and r_["max_abs_logit_diff"] <= 1.0})
    E2.append(r_)
E2_EAGER = None
if ATTN != "eager":
    try:
        assert set_attn("eager"), "could not switch to eager"
        E2_EAGER = [dict(e2_run(ids), prompt=name) for (name, _), ids in zip(C5A_PROMPTS, c5a_ids)]
    except Exception as e:
        E2_EAGER = {"error": type(e).__name__ + ": " + str(e)[:200]}
    finally:
        _ok = set_attn(ATTN)
    assert _ok and model.config._attn_implementation == ATTN, "could not restore the default attention"
    torch_mod.cuda.empty_cache()

G1 = []
BASE_TOKENS = []
for (name, _), ids in zip(C5A_PROMPTS, c5a_ids):
    st, past, new, ref_tokens = _single(), None, tuple(ids), []
    for _ in range(64):
        lg, past = st(new, past)
        tok = q8._greedy([float(v) for v in lg.tolist()])
        ref_tokens.append(tok); new = (tok,)
        if tok in set(EOS):
            break
    gen = rr.generate_base_tau(Bt, _single(), ids, EOS, TAU_EXCLUDED, len(ref_tokens))
    gen2 = rr.generate_base_tau(Bt, _single(), ids, EOS, TAU_EXCLUDED, len(ref_tokens))
    BASE_TOKENS.append(gen["token_ids"])
    G1.append({"prompt": name, "steps": len(ref_tokens), "identical": gen["token_ids"] == ref_tokens,
               "repeat_identical": gen2["token_ids"] == gen["token_ids"]
                                   and [s["tau_token"] for s in gen2["steps"]] == [s["tau_token"] for s in gen["steps"]],
               "tau_matched_steps": sum(bool(s["tau_matched"]) for s in gen["steps"]),
               "tau_ok": all(s["tau_token"] not in set(TAU_EXCLUDED) and s["tau_token"] != s["top1"] for s in gen["steps"])})
ids = c5a_ids[0]
_bs, _ss = _batch(), _single()
bp, sp, bn, sn = None, None, [tuple(ids)] * 32, tuple(ids)
b1_agree = b1_rows = 0
for _ in range(32):
    bl, bp = _bs(bn, bp); sl, sp = _ss(sn, sp)
    tok = int(sl.argmax()); b1_agree += int(int(bl[0].argmax()) == tok); b1_rows += int(bool((bl == bl[0:1]).all()))
    bn, sn = [(tok,)] * 32, (tok,)
del _bs, _ss, bp, sp, bl, sl
B1 = {"steps": 32, "agree_steps": b1_agree, "rows_identical_steps": b1_rows, "pass": b1_agree >= 29}

# M1
P_ids, chunk = tuple(c5a_ids[0]), tuple(BASE_TOKENS[0][:16])
with torch_mod.no_grad():
    out = model(input_ids=torch_mod.tensor([list(P_ids)], device=DEVICE), use_cache=True, logits_to_keep=1)
    out = model(input_ids=torch_mod.tensor([list(chunk)], device=DEVICE), past_key_values=out.past_key_values,
                use_cache=True, logits_to_keep=1)
last = out.logits[0, -1].float()
top3 = [int(v) for v in torch_mod.topk(last, 3).indices.tolist()]
st, past = _single(), None
for new in [P_ids] + [(t_,) for t_ in chunk]:
    tbt, past = st(tuple(new), past)
del past
tbt_top2 = torch_mod.topk(tbt, 2).values
chunk_cmp = {"argmax_agrees": top3[0] == int(tbt.argmax()), "max_abs_logit_diff": float((last - tbt).abs().max()),
             "token_by_token_gap": float(tbt_top2[0] - tbt_top2[1])}
big, how = S.expand_cache(out.past_key_values, 48)
refs = []
for x in top3:
    st, past, new, seqr = _single(), None, P_ids + chunk + (x,), []
    for _ in range(32):
        lg, past = st(new, past)
        t = int(lg.argmax()); seqr.append(t); new = (t,)
    refs.append(seqr)
bst = _batch()
rows = [(x,) for x in top3 for _ in range(16)]
agree = [0, 0, 0]; same = [0, 0, 0]
past = big
for s_ in range(32):
    lg, past = bst(rows, past)
    am = [int(v) for v in lg.argmax(dim=-1).tolist()]
    for g_ in range(3):
        grp = am[16 * g_:16 * (g_ + 1)]
        agree[g_] += int(grp[0] == refs[g_][s_]); same[g_] += int(len(set(grp)) == 1)
    rows = [(refs[g_][s_],) for g_ in range(3) for _ in range(16)]
del big, past, out, lg
torch_mod.cuda.empty_cache()
M1 = {"expand_method": how, "chunk_vs_token_by_token": chunk_cmp, "agree_steps_per_group": agree,
      "rows_identical_steps_per_group": same,
      "pass": chunk_cmp["max_abs_logit_diff"] <= 1.0 and all(a_ >= 29 for a_ in agree)}
SMOKE["checks"] = {"E2": E2, "E2_eager": E2_EAGER, "G1": G1, "B1": B1, "M1": M1,
                   "revision": "2026-10-17 12:15 JST: E2 teacher-forced count rule; G1 vs cached greedy + repeat"}
atomic_json(OUT / "SMOKE.json", SMOKE)
print(json.dumps(SMOKE["checks"], indent=1))
assert all(r["pass"] for r in E2), "E2 が合いません（ずれが大きい、または1位が合う歩が29未満）— Claude に見せてね"
assert all(r["identical"] and r["repeat_identical"] and r["tau_ok"] for r in G1), "G1 が合いません — Claude に見せてね"
assert B1["pass"], "B1 が合いません — Claude に見せてね"
print("E2 / G1 / B1 PASS", "| M1", "PASS" if M1["pass"] else "（不一致あり。記録しました。止めません）")

In [ ]:
# Cell 5 — timing (not scored).  After warm-up, on the first C5-A prompt with a ~500-token prefix built from its own
# greedy tokens: BASE (base path with w_tau, per step); PREFILL1 (one row, the whole prefix); CHUNK / CHUNK8 (one-row
# forward of 16 / 8 tokens on that cache); EXPAND[n] (copy and repeat the one-row cache to n rows); DECODE[n] (one
# decode step of n rows with the pure sampler of every row, as generate_batch draws); FWD[n] (the forward alone);
# PREFILL[n] (DP8's way: n identical rows, one forward over the prefix + one draw).  Out of memory at a size is
# recorded (larger sizes are skipped).  Memory probe at the longest cache (prompt + 511, n rows, 4 steps).  If the
# default attention is not eager, DECODE[48] and PREFILL[32] are also timed with eager attention.
from pork.ct1_forks_replacement import generate_batch
from pork import q4b_samplers as qb
sync = torch_mod.cuda.synchronize
no_eos = (-1,)
ids = tuple(c5a_ids[0])
rr.generate_base_tau(Bt, _single(), ids, no_eos, TAU_EXCLUDED, 8)          # warm-up
sync(); t0 = time.time(); g = rr.generate_base_tau(Bt, _single(), ids, no_eos, TAU_EXCLUDED, 64); sync()
per = {"BASE": (time.time() - t0) / 64}
filler = tuple(g["token_ids"]) * 16
pre500 = (ids + filler)[:len(ids) + 480]
streams = [d8.stream("alt", 0, 0, j) for j in range(max(S.BATCH_SIZES))]

def is_oom(e):
    return isinstance(e, torch_mod.cuda.OutOfMemoryError) or "out of memory" in str(e).lower()

def one_row_cache(prefix):
    with torch_mod.no_grad():
        o = model(input_ids=torch_mod.tensor([list(prefix)], device=DEVICE), use_cache=True, logits_to_keep=1)
    return o.past_key_values, int(o.logits[0, -1].argmax())

def decode_steps(cache, first_tok, n, warm, steps, sample=True):
    st = _batch()
    past, new = cache, [(first_tok,)] * n
    for s_ in range(warm + steps):
        if s_ == warm:
            sync(); t0 = time.time()
        lg, past = st(new, past)
        if sample:
            new = [(int(qb.pure_sample(Bt, lg[r_], float(streams[r_][s_]), 1.0)[0]),) for r_ in range(n)]
    sync()
    return (time.time() - t0) / steps

def timed_prefill(n, prefix):
    sync(); t0 = time.time()
    generate_batch(Bt, _batch(), prefix, streams[:n], no_eos, 1.0, 1); sync()
    return time.time() - t0

one_row_cache(pre500)                                                    # warm-up
sync(); t0 = time.time(); base1, tok1 = one_row_cache(pre500); sync(); per["PREFILL1"] = time.time() - t0
with torch_mod.no_grad():
    for L_, key in ((16, "CHUNK"), (8, "CHUNK8")):
        c, _m = S.expand_cache(base1, 1)
        nxt = list(filler[:L_])
        sync(); t0 = time.time()
        for _ in range(3):
            o = model(input_ids=torch_mod.tensor([nxt], device=DEVICE), past_key_values=c, use_cache=True, logits_to_keep=1)
            c = o.past_key_values
        sync(); per[key] = (time.time() - t0) / 3
        del c, o
per["DECODE"], per["FWD"], per["EXPAND"], per["PREFILL"] = {}, {}, {}, {}
OOM_AT = None
for n in S.BATCH_SIZES:
    try:
        sync(); t0 = time.time()
        big, _m = S.expand_cache(base1, n); sync()
        per["EXPAND"][str(n)] = time.time() - t0
        per["DECODE"][str(n)] = decode_steps(big, tok1, n, 2, 24)
        del big
        big, _m = S.expand_cache(base1, n)
        per["FWD"][str(n)] = decode_steps(big, tok1, n, 1, 8, sample=False)
        del big
    except Exception as e:
        if not is_oom(e):
            raise
        OOM_AT = n
        big = None
        torch_mod.cuda.empty_cache()
        break
    torch_mod.cuda.empty_cache()
    print(f"n={n}: decode {per['DECODE'][str(n)]*1000:.0f} ms/step (forward alone {per['FWD'][str(n)]*1000:.0f}), "
          f"copy {per['EXPAND'][str(n)]*1000:.0f} ms")
for n in (16, 32, 48):
    try:
        timed_prefill(n, pre500)
        per["PREFILL"][str(n)] = timed_prefill(n, pre500)
    except Exception as e:
        if not is_oom(e):
            raise
        torch_mod.cuda.empty_cache()
        break
    torch_mod.cuda.empty_cache()
print("BASE", round(per["BASE"] * 1000), "ms/step | PREFILL1", round(per["PREFILL1"], 2), "s | CHUNK",
      round(per["CHUNK"] * 1000), "ms | PREFILL[n]", {k: round(v, 2) for k, v in per["PREFILL"].items()})

# Memory probe at the longest cache: one row of prompt + 507 tokens, repeated to n rows, 4 decode steps (-> +511).
probe = (ids + filler * 2)[:len(ids) + 507]
MEM = {}
del base1
torch_mod.cuda.empty_cache()
for n in (48, 64, 96, 128):
    torch_mod.cuda.empty_cache(); torch_mod.cuda.reset_peak_memory_stats()
    try:
        pc, pt1 = one_row_cache(probe)
        big, _m = S.expand_cache(pc, n)
        del pc
        decode_steps(big, pt1, n, 0, 4)
        MEM[str(n)] = torch_mod.cuda.max_memory_allocated() / 1e9
        del big
    except Exception as e:
        if not is_oom(e):
            raise
        MEM[str(n)] = None
        pc = big = None
        torch_mod.cuda.empty_cache()
        break
    print(f"memory probe n={n}: peak {MEM[str(n)]:.1f} GB")
torch_mod.cuda.empty_cache()

SMOKE["timing"] = {"per": per, "oom_at": OOM_AT, "prefix_tokens": len(pre500), "memory_probe_gb": MEM,
                   "probe_tokens": len(probe) + 4, "eager": None}
atomic_json(OUT / "SMOKE.json", SMOKE)


EAGER = None
if ATTN != "eager":
    try:
        assert set_attn("eager"), "could not switch to eager"
        b1c, b1t = one_row_cache(pre500)
        big, _m = S.expand_cache(b1c, 48)
        EAGER = {"DECODE48": decode_steps(big, b1t, 48, 2, 24)}
        del big, b1c
        torch_mod.cuda.empty_cache()
        EAGER["PREFILL32"] = timed_prefill(32, pre500)
    except Exception as e:
        EAGER = {"error": type(e).__name__ + ": " + str(e)[:200]}
    finally:
        restored = set_attn(ATTN)
        torch_mod.cuda.empty_cache()
    EAGER["restored_default_attention"] = restored
    if not restored:
        print("⚠️ 注意の計算方法を元に戻せませんでした（記録します。止めません）")
SMOKE["timing"]["eager"] = EAGER
atomic_json(OUT / "SMOKE.json", SMOKE)
print(json.dumps(SMOKE["timing"], indent=1))

In [ ]:
# Cell 6 — how long the greedy solutions are under the study's prompt, on the four problems written for this smoke
# (not from any benchmark).  Counts only: tokens, EOS, the first answer token, readable, correct.
LENS = []
for (q, ans), ids in zip(S.OWN_PROBLEMS, own_ids):
    sync(); t0 = time.time()
    gen = rr.generate_base_tau(Bt, _single(), ids, EOS, TAU_EXCLUDED, d2.MAX_NEW_TOKENS)
    sync(); sec = time.time() - t0
    rec = dp.base_record(gen["token_ids"], gen["eos"], decode, "#### " + ans)
    p1 = [float(s["p1"]) for s in gen["steps"]]
    LENS.append({"tokens": len(gen["token_ids"]), "eos": gen["eos"], "answer_token": rec["answer_token"],
                 "readable": rec["prediction"] is not None, "correct": rec["correct"], "seconds_per_step": sec / max(1, len(gen["token_ids"])),
                 "steps_r_le_1e-4": sum(1 for v in p1 if 1 - v <= 1e-4), "steps_r_gt_0.05": sum(1 for v in p1 if 1 - v > 0.05)})
SMOKE["own_problem_lengths"] = LENS
atomic_json(OUT / "SMOKE.json", SMOKE)
print(json.dumps(LENS, indent=1))

In [ ]:
# Cell 7 — projection of the DP9 run (40 items, one worker on this GPU) for each design, way and length, at three
# factors (raw; x1.7 = DP8's realised ratio of actual to projected; x2.7 = the factor DP4-DP8 used).  Then done.
proj = S.projection(SMOKE["timing"]["per"], workers=1)
SMOKE["projection"] = proj
SMOKE["finished_jst"] = dt.datetime.now(dt.timezone(dt.timedelta(hours=9))).strftime("%Y%m%d_%H%M%S")
atomic_json(OUT / "SMOKE.json", SMOKE)
print("design            way       L    hours(raw / x1.7 / x2.7)     CU(x1.7)")
for r_ in proj:
    h = r_["hours"]
    if h is None:
        print(f"{r_['design']:<17} {r_['way']:<9} {r_['length']:<4} 入らない（メモリ）")
    else:
        print(f"{r_['design']:<17} {r_['way']:<9} {r_['length']:<4} {h['raw']:6.1f} / {h['dp8_realised']:6.1f} / {h['conservative']:6.1f}   {r_['cu']['dp8_realised']:6.1f}")
print("✅ 計測おわり。結果：DECISION_POINTS/dp9/SMOKE_" + STAMP + "/SMOKE.json")
notify('🎉 終わりました：DP9 の時間計測（Qwen2.5-7B・L4）。結果は dp9/SMOKE_' + STAMP, tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()